# Task 1 / OMD ベースライン: 事前学習モデルの Fine-tuning

`task1_baseline.ipynb`（特徴量＋ロジスティック回帰）と対をなす配布ベースラインです。
事前学習済み言語モデルを本タスクのデータで **fine-tuning** します。引用文脈と論文情報のペアを
1つのモデルに直接読ませて判定させる **cross-encoder（文ペア分類）** という構成で、
特徴量を人手で設計する代わりに、モデル自身にペアの照合を学習させます。

**使用モデル**: [sbintuitions/modernbert-ja-70m](https://huggingface.co/sbintuitions/modernbert-ja-70m)
（日本語 ModernBERT、パラメータ 70M）

**レギュレーションとの対応**
- パラメータ数 70M ≤ **100M 以下** ✓
- 学習時間は T4 GPU で1〜2分（規定: **Colab Pro A100 基準で総実行時間1時間以内**）✓（独自環境の学習済み重みは使わない）
- 学習データは**配布データのみ** ✓

**データの使い方**
```
train.jsonl        → fine-tuning の学習データ
dev_labeled.jsonl  → エポックごとの評価とモデル選択（ベスト重みの保持）
dev_leaderboard.jsonl → 最終セルで予測を出力して提出
```

**実行環境**: **GPU ランタイム必須**です（メニュー［ランタイム］→［ランタイムのタイプを変更］→
**T4 GPU**）。学習時間の目安は T4 GPU で1〜2分、CPU では数十分以上です。


In [ ]:
# 必要なライブラリ（初回のみ）
%pip install -q transformers sentencepiece scikit-learn

In [ ]:
# データの取得（Google Colab 用）: 配布リポジトリをクローンする。
# ローカルで配布リポジトリの中から実行している場合、このセルは何もしません。
![ -d data ] || [ -d ../data ] || git clone -q https://github.com/YANS-official/yans-2026-hackathon

In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import torch

# 再現性のためにシードを固定
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# 配布データの場所を自動で探す（./data → ../data → クローン先）
_candidates = ([Path(os.environ["YANS_DATA_DIR"])] if os.environ.get("YANS_DATA_DIR") else []) + [
    Path("data"), Path("../data"), Path("yans-2026-hackathon/data")]
DATA_DIR = next((p for p in _candidates if (p / "train.jsonl").exists()), None)
assert DATA_DIR is not None, "配布データ（data/）が見つかりません"
print(f"データディレクトリ: {DATA_DIR.resolve()}")

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train = read_jsonl(DATA_DIR / "train.jsonl")
dev = read_jsonl(DATA_DIR / "dev_labeled.jsonl")
lb = read_jsonl(DATA_DIR / "dev_leaderboard.jsonl")
papers = {p["paper_id"]: p for p in read_jsonl(DATA_DIR / "papers.jsonl")}
print(f"train={len(train)}  dev_labeled={len(dev)}  dev_leaderboard={len(lb)}")

## 1. モデルとトークナイザの準備

引用文脈（文A）と論文情報（文B: タイトル＋概要）を**文ペア**としてまとめて入力し、
2クラス（妥当/不適切）を出力するヘッドを載せて学習します。トークナイザの
`(文A, 文B)` 引数でこのペア入力を構成します。

### 1.1 本文チャンクの検索（retrieval）による文Bの拡張

`task1_finetune.ipynb` の元の設計では文B（論文情報）はタイトル＋概要のみですが、
論文の本文（`tex_content`）には引用文脈と直接対応する記述（提案手法の詳細、実験設定、
数値など）が含まれていることがあります。ここでは `scripts/extract_chunks.py` で
事前に分割した本文チャンク（`data/chunks.jsonl`）から、引用文脈と類似度の高い上位 N 件を
検索し、文Bに追加します。

検索には分類本体とは別の軽量な埋め込みモデル `cl-nagoya/ruri-v3-30m`（30M パラメータ）を
使います。分類モデル（`sbintuitions/modernbert-ja-70m`, 70M）とは別のモデルなので、
両方とも規定の **100M パラメータ以下** を満たします（1つのモデルに統合しているわけではない）。

In [ ]:
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

CHUNK_MODEL_NAME = "cl-nagoya/ruri-v3-30m"
TOP_N_CHUNKS = 3

chunks_by_paper = {}
with open(DATA_DIR / "chunks.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        chunks_by_paper.setdefault(rec["paper_id"], []).append(rec["text"])

chunk_model = SentenceTransformer(CHUNK_MODEL_NAME, device=device)
print("チャンク検索用モデル パラメータ数:",
      f"{sum(p.numel() for p in chunk_model.parameters())/1e6:.0f}M")

# 論文ごとにチャンクを埋め込む（引用文脈との類似度検索に使う）
chunk_embs_by_paper = {}
for pid, texts in chunks_by_paper.items():
    chunk_embs_by_paper[pid] = chunk_model.encode(texts, normalize_embeddings=True)

# 全レコードの引用文脈をまとめて埋め込み、各レコードについて対象論文のチャンクの中から
# 類似度上位 TOP_N_CHUNKS 件を検索してテキストに連結しておく（学習中に毎回埋め込み直さないよう
# 事前計算してキャッシュする）
def build_top_chunks_cache(records):
    contexts = [r["citation_context"] for r in records]
    ctx_embs = chunk_model.encode(contexts, normalize_embeddings=True, show_progress_bar=True, batch_size=64)
    cache = {}
    for r, ctx_emb in zip(records, ctx_embs):
        pid = r["cited_paper_id"]
        texts = chunks_by_paper.get(pid)
        embs = chunk_embs_by_paper.get(pid)
        if not texts or embs is None or len(embs) == 0:
            cache[r["id"]] = ""
            continue
        sims = embs @ ctx_emb
        top_idx = np.argsort(-sims)[:TOP_N_CHUNKS]
        cache[r["id"]] = " ".join(texts[i] for i in top_idx)
    return cache

top_chunks_cache = {}
for name, records in [("train", train), ("dev", dev), ("lb", lb)]:
    print(f"チャンク検索キャッシュを構築中: {name} ({len(records)}件) ...")
    top_chunks_cache.update(build_top_chunks_cache(records))

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "sbintuitions/modernbert-ja-70m"
MAX_LEN = 256      # 入力の最大トークン長（チャンク追加分を切り捨てないよう128から拡大）
BATCH_SIZE = 16
EPOCHS = 5
LR = 5e-5

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    print("警告: GPU がありません。CPU では学習に数十分以上かかります。")
    print("Colab では［ランタイム］→［ランタイムのタイプを変更］→ T4 GPU を選んでください。")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
print("計算デバイス:", device, "/ パラメータ数:", f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M")

def encode_batch(records):
    ctx = [r["citation_context"] for r in records]
    paper_text = [
        f"{papers[r['cited_paper_id']]['title']} {papers[r['cited_paper_id']]['abstract']} "
        f"{top_chunks_cache.get(r['id'], '')}"
        for r in records
    ]
    return tokenizer(ctx, paper_text, truncation=True,
                     max_length=MAX_LEN, padding=True, return_tensors="pt")

## 2. Fine-tuning

標準的な設定（AdamW・線形スケジューラ・5エポック）で学習します。
**各エポックの終わりに dev_labeled で評価し、最も良かった時点の重みを保持**します
（エポックを増やすと train への適合は進む一方、dev の性能は低下しうる＝過学習）。
学習時間は T4 GPU で1〜2分です。

In [ ]:
import copy

from sklearn.metrics import accuracy_score, f1_score
from transformers import get_linear_schedule_with_warmup

# このセルを再実行したときに「前回の学習済み重みからの続き」にならないよう、
# 毎回モデルと乱数シードを初期化し直す（何度実行しても同じ条件の学習になる）
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

@torch.no_grad()
def predict(records, batch_size=64):
    model.eval()
    preds = []
    for i in range(0, len(records), batch_size):
        enc = encode_batch(records[i:i + batch_size]).to(device)
        preds += model(**enc).logits.argmax(-1).cpu().tolist()
    return np.array(preds)

y_train = np.array([int(r["label"]) for r in train])
y_dev = np.array([int(r["label"]) for r in dev])

order = list(range(len(train)))
steps_per_epoch = (len(train) + BATCH_SIZE - 1) // BATCH_SIZE
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(0.1 * EPOCHS * steps_per_epoch), EPOCHS * steps_per_epoch)

# 各エポック終了時に dev_labeled で評価し、チェックポイント（重み）を保存する。
# 学習を進めるほど train には適合するが、dev の性能はどこかで頭打ちになる。
# 全エポックの学習曲線を確認した上で、検証 Accuracy が最大のエポックの重みを採用する
# （早期終了 early stopping の標準的な考え方）。
checkpoints = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    random.shuffle(order)
    total_loss = 0.0
    for i in range(0, len(order), BATCH_SIZE):
        batch = [train[j] for j in order[i:i + BATCH_SIZE]]
        enc = encode_batch(batch).to(device)
        labels = torch.tensor([int(r["label"]) for r in batch], device=device)
        loss = model(**enc, labels=labels).loss
        loss.backward()
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total_loss += float(loss)
    acc = accuracy_score(y_dev, predict(dev))
    print(f"epoch {epoch}: 平均loss={total_loss / steps_per_epoch:.3f}  dev Accuracy={acc:.3f}")
    checkpoints.append((acc, copy.deepcopy(model.state_dict())))

adopt_acc, adopt_state = max(checkpoints, key=lambda x: x[0])
model.load_state_dict(adopt_state)
print(f"\n採用: dev Accuracy={adopt_acc:.3f} のエポックの重み（採用規則: 検証 Accuracy 最大）")

## 3. 性能とモデルの傾向を確認する

`task1_baseline.ipynb` と同じ流れで、train / dev_labeled の性能と、正解・不正解の事例を確認します。
ベースライン（特徴量＋LR）の dev_labeled の値と見比べて、fine-tuning の効果を確かめましょう。

In [ ]:
pred_train = predict(train)
pred_dev = predict(dev)
print(f"[fine-tuning] train      : Accuracy={accuracy_score(y_train, pred_train):.3f}  "
      f"F1={f1_score(y_train, pred_train):.3f}")
print(f"[fine-tuning] dev_labeled: Accuracy={accuracy_score(y_dev, pred_dev):.3f}  "
      f"F1={f1_score(y_dev, pred_dev):.3f}")

def show_cases(records, golds, preds, correct, n=3):
    shown = 0
    for i, r in enumerate(records):
        if (preds[i] == golds[i]) != correct:
            continue
        paper = papers[r["cited_paper_id"]]
        print(f"--- {r['id']}  正解={golds[i]}  予測={preds[i]} ---")
        print("文脈:", r["citation_context"])
        print(f"論文: {paper['title']}（{paper['year']}）")
        print()
        shown += 1
        if shown >= n:
            return

print("\n========== 間違えた事例（ベースラインと同じ問題で間違えている？） ==========\n")
show_cases(dev, y_dev, pred_dev, correct=False, n=3)

## 4. 最終出力: dev_leaderboard への予測を保存する

In [ ]:
pred_lb = predict(lb)
with open("submission_task1_ft.jsonl", "w", encoding="utf-8") as f:
    for rec, p in zip(lb, pred_lb):
        f.write(json.dumps({"id": rec["id"], "prediction": int(p)}, ensure_ascii=False) + "\n")
print(f"submission_task1_ft.jsonl を書き出しました（{len(lb)}行 / 予測ラベル分布: "
      f"1が{int(pred_lb.sum())}問, 0が{len(lb) - int(pred_lb.sum())}問）")

## 5. 改善の方向性

1. **ハイパーパラメータ**: `MAX_LEN`（現在256）・`EPOCHS`・`LR`・`BATCH_SIZE`・
   `TOP_N_CHUNKS`（現在3）を dev_labeled で調整する。学習は seed や実行のたびに揺れるので、
   学習曲線を確認し、崩れた run は再実行する
2. **入力の設計**: 本ノートブックでは概要に加え `tex_content`（本文）由来のチャンクのうち
   引用文脈と類似度の高い上位 N 件を検索して文Bに追加済み（`task1_finetune.ipynb` からの拡張）。
   検索モデルの変更や、チャンクの分割サイズ（`scripts/extract_chunks.py`）の調整でさらに改善しうる
3. **モデルの選択**: 別系統の事前学習モデル（**100M 以下**の規定内で）を試す。
   事前学習の目的・言語・系列長上限によって、同じレシピでも到達値は大きく変わる
4. **組み合わせ**: fine-tuning の予測確率を特徴量ベースラインに加える（スタッキング）

**提出前のチェック**: 各モデルが 100M 以下か（分類モデル70M＋チャンク検索用モデル30Mは
別モデルなのでそれぞれ規定を満たす）・モデル構築の総実行時間が規定内（Colab Pro A100 基準で
1時間以内）か・使ったデータが配布物（そのまま/加工）または自作のみか、を確認してください。
